# Model Results

Fits and evalutes the Bayesian model.

## Imports

In [22]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

In [23]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [24]:
from src.config import PROCESSED_DATA_DIR, RANDOM_SEED, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model.model import BayesianPoissonModel, BayesianPoissonModelConfig, SVIConfig

## Evaluation

### Load Training and Testing Data

In [25]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 3040 matches
Test: 760 matches


### Bayesian Model

In [ ]:
bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = False,
        ablate_defense = False,
        ablate_home_team_advantage = False,
        num_posterior_samples_per_match = 25,
    )
).fit(train_df)
bayesian_model_preds = bayesian_model.predict(test_df)

Step 100: Loss = 9359.277
Step 200: Loss = 9192.884
Step 300: Loss = 9184.015
Step 400: Loss = 9190.682
Step 500: Loss = 9169.601
Step 600: Loss = 9168.463
Step 700: Loss = 9171.383
Step 800: Loss = 9167.521
Step 900: Loss = 9169.867
Step 1000: Loss = 9167.339
Step 1100: Loss = 9167.054


In [ ]:
bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.623498,0.215093,0.161409
1,West Ham,Aston Villa,A,0.372980,0.256246,0.370774
2,Nott'm Forest,Bournemouth,D,0.536085,0.224819,0.239097
3,Newcastle,Southampton,H,0.544045,0.235536,0.220419
4,Everton,Brighton,A,0.434198,0.275700,0.290102


In [ ]:
pred_labels = bayesian_model_preds[["ProbHomeWin",
                                    "ProbDraw",
                                    "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                          "ProbDraw": "D",
                                                                          "ProbAwayWin": "A"})
accuracy = (pred_labels == bayesian_model_preds["Result"]).mean()
print(f"Bayesian Model Accuracy: {accuracy:.3f}")

Bayesian Model Accuracy: 0.451


### Probabilistic Evaluation Metrics

Beyond plain accuracy, we score the full predicted distribution with three more metrics:
Lower is better for all 3.
1. Log loss
2. Ranked Probability Score (RPS)
3. Brier score (squared error of prob vector)

In [ ]:
from src.metrics import evaluate

pd.DataFrame({"Bayesian Model": evaluate(bayesian_model_preds)}).T.round(4)

NameError: name 'bayesian_model_preds' is not defined

### Save Evaluation Results

In [ ]:
model_table_output_directory = (TABLES_DIR / "model")
model_table_output_directory.mkdir(parents = True, exist_ok = True)

bayesian_model_preds.to_csv(model_table_output_directory / "bayesian_model_predictions.csv",
                            index = False)